<a href="https://colab.research.google.com/github/tmzt/TrainingExperiments/blob/main/Highbay/Local/HighbaySchemaProseFinetune2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Highbay schema/prose fine-tune - v2

Same spine as v1: unsloth on a Colab T4, data from Drive, LoRA + GGUF out.
Six things changed, each because it was measured rather than suspected.

### 1. The target is the whole AST, not just `ui_prompt`

v1's `formatting_prompts_func` put `output_ast["ui_prompt"]` in the assistant
turn, so it trained the model to answer *"Shall I create a Water Logs table to
track your daily intake in ounces?"* - the suggestion-chip sentence.

**`ui_prompt` is a FIELD OF the AST.** Targeting the whole AST yields the prose
for free and the machine-readable structure as well, which is what the Genius
view actually consumes (GENIUS_PLAN section 3 uses `ui_prompt` for the chip;
everything else in the view is driven by `intent_type`, `schema_mutations` and
`pipeline_ast`). Training on the prose alone throws the structure away.

`TARGET = "ast" | "ui_prompt"` if you want v1's behaviour back.

### 2. Nothing overwrites your Drive data

v1's normalization cell did `open(dataset_path, 'w')` on the source file, in
place, and rewrote `payload` and `conditions[].value` as JSON **strings**. That
is destructive and it corrupts the ground truth: after one run,
`payload` is no longer an object, so a model trained on the AST learns the
stringified form. v2 normalizes **in memory only** and never writes the input.

### 3. Separate input and output sides - the AST's contents are not the frame's business

v1 fed the raw AST to `load_dataset("json", ...)`, which asks Arrow to infer a
column type per AST field. That is what forced v1's debug and normalize cells.

The frame does not need any of it. **There are two sides - an input string and
an output string** - and the output's internal shape is payload, not schema. So
v2 reads the JSON itself, serializes the AST once, and everything downstream
sees `{"text": str}`. A `Dataset`/DataFrame is perfectly fine at that point;
what is not fine is exploding an AST into typed columns.

For the record, what that inference did, measured:

* merged, Arrow refuses outright - `cannot mix struct and non-struct, non-null
  values`;
* on `mobile_data_prompts.jsonl` alone it does **not** error - it unifies the
  two mutation shapes into one 5-field struct, backfills nulls, and only
  **46/92** ASTs survive the round trip.

### 3b. The corpus is a generated training set, and it contradicts itself

These files came out of Gemini, so their shape conflates things a hand-authored
schema would not. One case is worth fixing before training rather than after,
because **the corpus currently teaches an inconsistency**.

`action.payload` looks like a `str | dict` union. It is not one - it is
determined by the action:

| action | payload | n |
|---|---|---|
| `SEND_NOTIFICATION` | message string | 28/28 |
| `CALCULATE` | description string | 20/20 |
| `UPDATE_RECORD` | field -> value **map** | 23/23 |

...but the generator could not decide how to write that map. 8 records use an
object, `{"status": "Warm"}`, and 15 use a *stringified* object,
`"{\"status\": \"Archived\"}"`. Identical meaning, two spellings, no rule
the model can learn - so at 114 training records it is being taught to guess.

`normalize_payload` below parses the stringified maps back, **in memory**, so
all 23 agree. `conditions[].value` (`str` x44, `int` x4) is left alone: that
one is real - `total_amount > 500` beside string comparisons.

### 4. Both corpora

`mobile_data_prompts.jsonl` (92) and `prompts.jsonl` (50) are shape-identical -
same keys, same two mutation shapes, same trigger/action vocabularies (mobile
adds `ON_DELETE`) - so they merge to **142** with no reconciliation, 142/142
well-formed. v1 used only the first.

Note `prompts.jsonl` is a pretty-printed JSON **array** despite the `.jsonl`
name; reading it line-by-line does not error, it silently yields
`schema_mutations` fragments that look like records.

### 5. A held-out split and a real metric

v1 ran `max_steps = 60` with no eval set and no measurement, so there was no way
to tell whether it had worked. v2 holds out a stratified split, runs the eval
**before** training as a baseline, and reports exact-AST match. A 3B instruct
model already emits plausible JSON, so the post-training number alone says
nothing.

### 6. A system message

v1's conversations were user -> assistant with no system turn. The model is
being asked for strict JSON; say so.

### Kept from v1

The `torchao` pin, unsloth's `FastLanguageModel` and `use_gradient_checkpointing
= "unsloth"`, `fp16 = not is_bf16_supported()` (correct: a T4 is Turing and has
no bf16), Drive mount, and `save_pretrained_gguf`.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Pin torchao to a version compatible with PEFT (kept from v1)
!pip install torchao==0.18.0

In [ ]:
# 2. Install unsloth and dependencies (kept from v1)
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# 3. Runtime check - which dtype does this card actually support?
import torch, subprocess

print("GPU:", subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."

BF16 = torch.cuda.is_bf16_supported()
print(f"compute capability {torch.cuda.get_device_capability()}  bf16={BF16}")
if not BF16:
    print("T4 path: fp16. Turing has no bfloat16 - anything hardcoding bf16=True fails.")

In [ ]:
# 4. Load 4-bit Llama-3.2-3B-Instruct + LoRA (kept from v1)
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import os, json, random, collections

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# chatml, WITHOUT v1's mapping= argument: that mapping ({"role":"from",
# "content":"value", ...}) is for ShareGPT-shaped data. We build role/content
# messages ourselves, so it applied to nothing.
tokenizer = get_chat_template(tokenizer, chat_template = "chatml")

## 5. Load the corpora - read the JSON ourselves

`utf-8-sig` because `mobile_data_prompts.jsonl` carries a BOM: with plain
`utf-8` the **first** record and only the first fails to parse, which reads like
one stray malformed line rather than an encoding problem.

**Nothing here writes to Drive.**

In [ ]:
# 5. Read both corpora. No load_dataset on the raw AST - see the header.
DRIVE_DIR = "/content/drive/MyDrive/Training Data"
CORPUS_FILES = ["mobile_data_prompts.jsonl", "prompts.jsonl"]
EXTRA_CORPUS_PATH = None      # anything you generate later

def load_corpus(text):
    """Accept a JSON array or true JSONL. prompts.jsonl is an ARRAY despite its
    name, and reading it line-by-line does not error - it yields
    schema_mutations fragments that look like records - so try the array first."""
    text = text.strip()
    try:
        data = json.loads(text)
        return data if isinstance(data, list) else [data]
    except json.JSONDecodeError:
        return [json.loads(l) for l in text.splitlines() if l.strip()]

records = []
for fn in CORPUS_FILES:
    path = os.path.join(DRIVE_DIR, fn)
    try:
        got = load_corpus(open(path, encoding="utf-8-sig").read())   # BOM
    except FileNotFoundError:
        print("  skipped (not found):", path)
        continue
    print(f"  {fn}: {len(got)} records")
    records += got

if EXTRA_CORPUS_PATH:
    got = load_corpus(open(EXTRA_CORPUS_PATH, encoding="utf-8-sig").read())
    print(f"  {EXTRA_CORPUS_PATH}: {len(got)} records")
    records += got

seen, deduped = set(), []
for r in records:
    if r["user_input"] not in seen:
        seen.add(r["user_input"]); deduped.append(r)
if len(deduped) != len(records):
    print(f"  dropped {len(records) - len(deduped)} duplicate user_inputs")
records = deduped

assert records, f"no records loaded - is the data in {DRIVE_DIR}?"
print(f"{len(records)} records")

In [ ]:
# 6. Contract: vocabularies read off the corpus, not invented.
INTENT_TYPES  = {"PIPELINE", "SCHEMA_SUGGESTION"}
TRIGGER_TYPES = {"ON_CREATE", "ON_UPDATE", "ON_DELETE", "SCHEDULED"}
ACTION_TYPES  = {"SEND_NOTIFICATION", "UPDATE_RECORD", "CALCULATE"}
COLUMN_TYPES  = {"NUMBER", "DATE", "STRING", "BOOLEAN", "RELATION"}
# TWO mutation shapes: reading m["table"] skips every table creation, quietly,
# because .get returns None rather than raising.
MUTATION_KEYS = {
    "CREATE_TABLE": {"action", "name"},
    "ADD_COLUMN":   {"action", "table", "column_name", "type"},
}

def mutation_table(m):
    return m.get("table") or m.get("name")

def canonical(ast):
    """ONE string form per AST: training target and eval comparison are the same
    definition rather than two that drift."""
    return json.dumps(ast, sort_keys=True, separators=(",", ":"))

def validate(ast):
    problems = []
    if not isinstance(ast, dict):
        return ["not an object"]
    it = ast.get("intent_type")
    if it not in INTENT_TYPES:
        problems.append(f"intent_type={it!r}")
    if it == "PIPELINE":
        p = ast.get("pipeline_ast")
        if not isinstance(p, dict):
            problems.append("PIPELINE without pipeline_ast")
        else:
            trig, act = p.get("trigger"), p.get("action")
            if not isinstance(trig, dict): problems.append("trigger missing")
            elif trig.get("type") not in TRIGGER_TYPES:
                problems.append(f"trigger.type={trig.get('type')!r}")
            if not isinstance(act, dict): problems.append("action missing")
            elif act.get("type") not in ACTION_TYPES:
                problems.append(f"action.type={act.get('type')!r}")
    if it == "SCHEMA_SUGGESTION":
        muts = ast.get("schema_mutations")
        if not isinstance(muts, list) or not muts:
            problems.append("SCHEMA_SUGGESTION without schema_mutations")
        else:
            for j, m in enumerate(muts):
                want = MUTATION_KEYS.get(m.get("action"))
                if want is None:
                    problems.append(f"mutation[{j}].action={m.get('action')!r}"); continue
                # Exact key set BOTH ways: an extra key is the Arrow-backfill
                # corruption, so this catches it too.
                if set(m) != want:
                    problems.append(f"mutation[{j}] keys {sorted(set(m))} != {sorted(want)}")
                elif m["action"] == "ADD_COLUMN" and m.get("type") not in COLUMN_TYPES:
                    problems.append(f"mutation[{j}].type={m.get('type')!r}")
    return problems

bad = [(i, p) for i, p in ((i, validate(r["output_ast"])) for i, r in enumerate(records)) if p]
print(f"{len(records) - len(bad)}/{len(records)} well-formed")
for i, p in bad[:8]:
    print("  record", i, p)

tables = collections.Counter()
for r in records:
    a = r["output_ast"]
    if a.get("pipeline_ast"):
        tables[a["pipeline_ast"].get("trigger", {}).get("table")] += 1
    for m in (a.get("schema_mutations") or []):
        tables[mutation_table(m)] += 1     # NOT m["table"]
print("intent_type:", dict(collections.Counter(
    r["output_ast"].get("intent_type") for r in records)))
print(f"{len([t for t in tables if t])} distinct tables")
for want in ("Project", "Expense", "Budget"):
    hits = sorted(t for t in tables if t and want.lower() in t.lower())
    print(f"  {want:8} -> {hits or 'ABSENT - no training data for this table'}")

## 7. ChatML

Each record becomes three messages. Arrow only ever sees `{"text": str}` from
here on, which is why none of v1's schema trouble can recur.

`TARGET` picks what the assistant turn contains. `"ast"` is the default because
`ui_prompt` is a field of the AST - targeting the AST gives you the prose *and*
the structure, while targeting the prose throws the structure away.

In [ ]:
# 7. Records -> ChatML text
TARGET = "ast"          # "ast" (default) | "ui_prompt" (v1's behaviour)

SYSTEM = (
    "You convert a user's plain-language automation request into a strict JSON "
    "AST. Reply with JSON only - no prose, no code fences."
)
SYSTEM_PROSE = (
    "You restate a user's plain-language automation request as a short "
    "confirming question. Reply with one sentence."
)

def normalize_payload(ast):
    """Make the corpus agree with itself. IN MEMORY ONLY - v1 rewrote the Drive
    file in place, and in the opposite direction.

    UPDATE_RECORD's payload is a field->value map in 23/23 records, but the
    generator wrote it as an object 8 times and as a STRINGIFIED object 15
    times. Two spellings of one meaning is not a rule a model can learn, so
    parse the strings back and let the object form win.

    v1 normalized the other way - json.dumps'ing the objects INTO strings - and
    wrote that to disk, which makes the stringified form the ground truth.
    Direction matters: the map is the meaning, the string is an artifact.
    """
    p = (ast.get("pipeline_ast") or {}).get("action")
    if p and isinstance(p.get("payload"), str):
        try:
            decoded = json.loads(p["payload"])
            if isinstance(decoded, dict):     # only maps; a message stays a message
                p = dict(p, payload=decoded)
                ast = dict(ast, pipeline_ast=dict(ast["pipeline_ast"], action=p))
        except json.JSONDecodeError:
            pass                              # a plain message, as intended
    return ast

def answer_for(record):
    if TARGET == "ui_prompt":
        return record["output_ast"].get("ui_prompt", "")
    return canonical(normalize_payload(record["output_ast"]))

def to_chatml(record):
    return {"messages": [
        {"role": "system",    "content": SYSTEM_PROSE if TARGET == "ui_prompt" else SYSTEM},
        {"role": "user",      "content": record["user_input"]},
        {"role": "assistant", "content": answer_for(record)},
    ]}

conversations = [to_chatml(r) for r in records]

if TARGET == "ast":
    recovered = [json.loads(c["messages"][2]["content"]) for c in conversations]
    assert all(canonical(a) == canonical(normalize_payload(r["output_ast"]))
               for a, r in zip(recovered, records)), "an AST changed through ChatML"
    print(f"all {len(records)} ASTs recovered byte-identical from the assistant turn")
    shapes = collections.Counter(
        type(((normalize_payload(r["output_ast"]).get("pipeline_ast") or {})
              .get("action") or {}).get("payload")).__name__
        for r in records
        if ((r["output_ast"].get("pipeline_ast") or {}).get("action") or {}).get("type")
           == "UPDATE_RECORD")
    print("UPDATE_RECORD payload types after normalization:", dict(shapes),
          "(was {'str': 15, 'dict': 8})")

CHATML_PATH = os.path.join(DRIVE_DIR, "chatml.jsonl")
with open(CHATML_PATH, "w", encoding="utf-8") as fh:     # a NEW file, not the input
    for c in conversations:
        fh.write(json.dumps(c, ensure_ascii=False) + "\n")
print("wrote", CHATML_PATH)
print(json.dumps(conversations[0], indent=2, ensure_ascii=False)[:500], "...")

In [ ]:
# 8. Stratified split, fixed seed - with a corpus this small the split dominates
#    the numbers, so do not change SEED between runs you mean to compare.
SEED, EVAL_FRACTION = 3407, 0.2

by_intent = collections.defaultdict(list)
for r in records:
    by_intent[r["output_ast"].get("intent_type")].append(r)

train_recs, eval_recs = [], []
rng = random.Random(SEED)
for intent, group in sorted(by_intent.items()):
    group = group[:]; rng.shuffle(group)
    cut = max(1, round(len(group) * EVAL_FRACTION))
    eval_recs += group[:cut]; train_recs += group[cut:]
rng.shuffle(train_recs); rng.shuffle(eval_recs)
print(f"train {len(train_recs)}  eval {len(eval_recs)}")

def to_text(record):
    return tokenizer.apply_chat_template(
        to_chatml(record)["messages"], tokenize = False, add_generation_prompt = False)

from datasets import Dataset
# Every column is a string here, so Arrow is safe - which was the whole point.
train_ds = Dataset.from_list([{"text": to_text(r)} for r in train_recs])
print(train_ds[0]["text"][:400], "...")

## 9. Baseline before training

Run the eval **first**. A 3B instruct model already produces plausible JSON, so
without this number the post-training one cannot be read.

In [ ]:
# 9. Eval: exact match on the canonical AST
def prompt_for(user_input):
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROSE if TARGET == "ui_prompt" else SYSTEM},
         {"role": "user",   "content": user_input}],
        tokenize = False, add_generation_prompt = True)

@torch.no_grad()
def predict(user_input, max_new_tokens = 512):
    ids = tokenizer(prompt_for(user_input), return_tensors = "pt",
                    add_special_tokens = False).to(model.device)
    out = model.generate(**ids, max_new_tokens = max_new_tokens, do_sample = False,
                         pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:],
                            skip_special_tokens = True).strip()

def parse_ast(text):
    """Models like to wrap JSON in fences or trail prose."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        text = text[4:] if text.lower().startswith("json") else text
    start = text.find("{")
    if start < 0: return None
    depth = 0
    for i, ch in enumerate(text[start:], start):
        depth += (ch == "{") - (ch == "}")
        if depth == 0:
            try: return json.loads(text[start:i + 1])
            except json.JSONDecodeError: return None
    return None

def evaluate(dataset, label):
    if TARGET != "ast":
        print(f"--- {label}: TARGET is {TARGET!r}; exact-AST match does not apply ---")
        for r in dataset[:3]:
            print("  in :", r["user_input"][:70])
            print("  out:", predict(r["user_input"], 128)[:120])
        return 0.0
    n = len(dataset); parsed = exact = valid = intent_ok = 0
    misses = []
    for r in dataset:
        got, want = parse_ast(predict(r["user_input"])), r["output_ast"]
        if got is None:
            misses.append((r["user_input"], "unparseable")); continue
        parsed += 1
        if not validate(got): valid += 1
        if canonical(got) == canonical(want): exact += 1
        else: misses.append((r["user_input"], canonical(got)[:140]))
        intent_ok += got.get("intent_type") == want.get("intent_type")
    print(f"--- {label}  (n={n}) ---")
    print(f"  parseable JSON  {parsed}/{n}")
    print(f"  schema-valid    {valid}/{n}")
    print(f"  EXACT AST       {exact}/{n}   <- the number that matters")
    print(f"  intent_type     {intent_ok}/{n}")
    for inp, got in misses[:5]:
        print(f"    miss {inp[:55]!r}\n         -> {got}")
    return exact / n if n else 0.0

FastLanguageModel.for_inference(model)
baseline = evaluate(eval_recs, "BASELINE (no fine-tuning)")

In [ ]:
# 10. Train
from trl import SFTTrainer
from transformers import TrainingArguments

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    train_dataset = train_ds,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # v1 used max_steps=60 regardless of corpus size. Epochs scale with the
        # data, which matters now that both corpora are merged.
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not BF16,          # T4
        bf16 = BF16,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = SEED,
        output_dir = "outputs",
        report_to = [],
    ),
)

# Loss on the ANSWER only. Without this most of the gradient goes on
# reproducing the question, which on a corpus this size is most of the signal.
try:
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(
        trainer,
        instruction_part = "<|im_start|>user\n",
        response_part    = "<|im_start|>assistant\n",
    )
    print("training on responses only")
except Exception as e:
    print("train_on_responses_only unavailable, training on the full sequence:", e)

trainer_stats = trainer.train()

In [ ]:
# 11. Eval after training, against the baseline
FastLanguageModel.for_inference(model)
tuned = evaluate(eval_recs, "AFTER FINE-TUNING")
if TARGET == "ast":
    print(f"\nexact-AST  baseline {baseline:.1%}  ->  tuned {tuned:.1%}"
          f"   (delta {tuned - baseline:+.1%})")
    _ = evaluate(train_recs[:10], "TRAIN SUBSET (memorization check)")
    print("\nNear-perfect on train while eval lags = memorization, which is the "
          f"expected shape at {len(train_recs)} records. The fix is more corpus, "
          "not more epochs.")

In [ ]:
# 12. Save to Drive (kept from v1)
OUT = os.path.join(DRIVE_DIR, "lora_model_v2")
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
print("adapter ->", OUT)

EXPORT_GGUF = True
if EXPORT_GGUF:
    model.save_pretrained_gguf(
        os.path.join(DRIVE_DIR, "model_v2_q4_k_m"), tokenizer,
        quantization_method = "q4_k_m")
    print("gguf -> done")